# emogist v2: LoRA training (Qwen3.5 2B or 4B)

This notebook trains a new **LoRA adapter** on the base Qwen3.5 2B (or 4B), using the v2 dataset from step 9. The v1 adapter is not used.

It is run **twice**, once per training set, in separate Colab sessions:

- **main**: English, French and Spanish (6,000 examples)
- **control**: English and French only, the same English and French examples as main (4,000)

Comparing the two shows whether adding Spanish helps or hurts English and French.

**Before you start:** in Colab, open **Runtime > Change runtime type**, select **T4 GPU**, and save. Then run the cells from top to bottom, one at a time (Shift+Enter).

**If Colab disconnects:** reconnect, select the T4 GPU again, and run every cell from the top. Step 7 continues from the last finished epoch saved in your Google Drive.

## 1. Install the libraries

Same versions as the v1 training (step 4), so only the data changes. This takes a few minutes.

If Colab asks you to restart the session afterwards, accept, then continue from step 2.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

## 2. Settings

**Set `RUN` to `"main"` or `"control"`, and `MODEL_SIZE` to `"2b"` or `"4b"`,** before running this cell. Everything else stays the same as v1 so the runs can be compared:
3 epochs, batch 4 with 2 accumulation steps, learning rate 0.0002, LoRA rank 16.

**4B only:** the model is about twice as large (9.3 GB), so training takes about twice as long. If step 7 stops with a "CUDA out of memory" error, restart the session, set `BATCH_SIZE = 2` and `ACCUMULATION_STEPS = 4` (still one update every 8 examples), and run again from step 2.

The instruction no longer says "short": v2 wants the length to come from the examples.

In [ ]:
import os
import torch

RUN = "main"                    # "main" (EN+FR+ES) or "control" (EN+FR)
MODEL_SIZE = "2b"               # "2b" or "4b"

LANGUAGES = {"main": ("en", "fr", "es"), "control": ("en", "fr")}[RUN]
BASE_MODEL = {"2b": "unsloth/Qwen3.5-2B", "4b": "unsloth/Qwen3.5-4B"}[MODEL_SIZE]
RUN_NAME = f"qwen{MODEL_SIZE}_v2_{RUN}"
TRAIN_FILE = "train_main.jsonl"   # control uses its English and French rows
DEV_FILE = "dev.csv"
DRIVE_DIR = f"/content/drive/MyDrive/emogist/run_{RUN_NAME}"
CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints"

EPOCHS = 3
BATCH_SIZE = 4
ACCUMULATION_STEPS = 2          # one update every 8 examples
LEARNING_RATE = 2e-4
LORA_RANK = 16
MAX_LENGTH = 256                # longest v2 example is 118 tokens
SEED = 3407
VALIDATION_SHARE = 0.1
SPLIT_SEED = 42

SYSTEM_PROMPT = "Convert the sentence into an emoji sentence that keeps its gist. Answer with emojis only."

assert torch.cuda.is_available(), "No GPU: use Runtime > Change runtime type > T4 GPU"
print("Run:", RUN_NAME, "languages:", LANGUAGES)
print("GPU:", torch.cuda.get_device_name(0))

## 3. Upload the data and connect Google Drive

Run this cell, then choose these two files from your repo:

- `09_v2_dataset_and_labels/train_main.jsonl`: the sentences and labels to learn from (both runs use this file)
- `08_v2_spanish_and_eval_sets/sets/dev.csv`: 300 dev sentences the model never trains on, used to score it afterwards

Colab then asks for access to your Google Drive. Checkpoints go to `MyDrive/emogist/`, so a disconnect doesn't lose the run.

In [ ]:
from google.colab import drive, files

needed = [TRAIN_FILE, DEV_FILE]
missing = [name for name in needed if not os.path.exists(name)]
if missing:
    print("Please select:", missing)
    files.upload()
missing = [name for name in needed if not os.path.exists(name)]
assert not missing, f"Still missing: {missing}"
print("Data ready:", needed)

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Checkpoints go to", DRIVE_DIR)

## 4. Load the base model

This downloads the original Qwen3.5 model (2B or 4B), not the v1 model.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)
# Qwen3.5 can also read images, so this may be a "processor" wrapping the text tokenizer.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
print("Model loaded")

## 5. Prepare the examples

The **validation set** (10%) is chosen the same way in both runs: 10% of each language and source (real or AI), with a fixed seed, from the full file.
The control run then drops the Spanish rows from both parts. So control trains on exactly the same English and French examples as main, and validates on the same English and French examples.

In [ ]:
import json
import random
from collections import defaultdict
from datasets import Dataset

def to_messages(sentence, emojis=None):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": sentence}]
    if emojis is not None:
        messages.append({"role": "assistant", "content": emojis})
    return messages

def to_text(row):
    text = tokenizer.apply_chat_template(to_messages(row["sentence"], row["emojis"]),
                                         tokenize=False, enable_thinking=False)
    return {"text": text}

with open(TRAIN_FILE, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]
assert len(rows) == 6000, f"Expected 6,000 rows, found {len(rows)}: is this the step 9 train_main.jsonl?"

# Same split for both runs: 10% of each (language, source) group.
groups = defaultdict(list)
for row in rows:
    groups[(row["lang"], row["source"])].append(row["id"])
rng = random.Random(SPLIT_SEED)
validation_ids = set()
for key in sorted(groups):
    ids = sorted(groups[key])
    rng.shuffle(ids)
    validation_ids.update(ids[:round(len(ids) * VALIDATION_SHARE)])

keep = [{"sentence": r["sentence"], "emojis": r["emojis"], "lang": r["lang"], "id": r["id"]}
        for r in rows if r["lang"] in LANGUAGES]
train_data = Dataset.from_list([r for r in keep if r["id"] not in validation_ids]).map(to_text)
validation_data = Dataset.from_list([r for r in keep if r["id"] in validation_ids]).map(to_text)
print(f"{len(train_data)} training examples, {len(validation_data)} validation examples")
for lang in LANGUAGES:
    print(f"  {lang}: {sum(l == lang for l in train_data['lang'])} train, {sum(l == lang for l in validation_data['lang'])} validation")
print()
print(train_data[0]["text"])

## 6. Attach the LoRA adapter

The 2 billion original weights are frozen. Only the small adapter trains.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_LENGTH,
)
model.print_trainable_parameters()

## 7. Train

The table shows the **training loss** regularly and the **validation loss** after each epoch. Lower is better.

- Both fall: it's learning.
- Training falls while validation rises: it's overfitting.
- Neither moves, or a value becomes `nan`: something is wrong, so stop and send me the output.

Each epoch is saved to Google Drive. At the end, the notebook keeps the epoch with the **lowest validation loss**. Run this step once per session.
On a T4, the 2B took 25 minutes for main and 18 for control; expect about twice that for the 4B.

In [ ]:
from trl import SFTTrainer, SFTConfig

# Rerunning this cell would keep training the same model and spoil the comparison.
assert "trainer" not in globals(), "Already trained: use Runtime > Restart session, then run from step 2"

import hashlib
from transformers.trainer_utils import get_last_checkpoint

# A resume must continue this exact run: same data, same split, same settings.
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
run_config = {"run": RUN_NAME, "languages": list(LANGUAGES), "base_model": BASE_MODEL, "epochs": EPOCHS,
              "batch_size": BATCH_SIZE, "accumulation_steps": ACCUMULATION_STEPS, "learning_rate": LEARNING_RATE,
              "lora_rank": LORA_RANK, "max_length": MAX_LENGTH, "seed": SEED, "system_prompt": SYSTEM_PROMPT,
              "validation_share": VALIDATION_SHARE, "split_seed": SPLIT_SEED,
              "train_sha256": hashlib.sha256(open(TRAIN_FILE, "rb").read()).hexdigest()}
config_file = f"{CHECKPOINT_DIR}/run_config.json"
if os.path.exists(config_file):
    with open(config_file) as f:
        assert json.load(f) == run_config, f"{CHECKPOINT_DIR} belongs to a different run: rename it or change RUN_NAME"
else:
    with open(config_file, "w") as f:
        json.dump(run_config, f, indent=1)
resume_from = get_last_checkpoint(CHECKPOINT_DIR)
print("Resuming from", resume_from) if resume_from else print("Starting from epoch 0")

new_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=validation_data,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_LENGTH,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=10,
        lr_scheduler_type="linear",
        weight_decay=0.001,
        optim="adamw_8bit",
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=EPOCHS,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=SEED,
        output_dir=CHECKPOINT_DIR,
        report_to="none",
    ),
)
result = new_trainer.train(resume_from_checkpoint=resume_from)
trainer = new_trainer

# One validation per epoch, no more: this catches a run that was trained twice.
evals = [e for e in trainer.state.log_history if "eval_loss" in e]
assert len(evals) == EPOCHS, f"Expected {EPOCHS} validations, found {len(evals)}: stop and send me the output"
best = min(evals, key=lambda e: e["eval_loss"])
for e in evals:
    print(f"Epoch {e['epoch']:.0f}: validation loss {e['eval_loss']:.4f}" + ("  <- kept" if e is best else ""))
print(f"This session trained for {result.metrics['train_runtime'] / 60:.1f} minutes")

## 8. Try it

A quick look before the real scoring. Feel free to change these sentences.

In [ ]:
assert "trainer" in globals(), "Step 7 has not finished in this session"
FastLanguageModel.for_inference(model)

def to_emojis(sentence):
    prompt = tokenizer.apply_chat_template(to_messages(sentence), tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
    inputs = text_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=60, do_sample=False, use_cache=True,
                                eos_token_id=text_tokenizer.eos_token_id,
                                pad_token_id=text_tokenizer.pad_token_id)
    answer = output[0][inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(answer, skip_special_tokens=True).strip()

for sentence in ["I lost my wallet on the bus this morning.",
                 "Ma sœur m'a dit qu'elle arrivait demain.",
                 "¿Sabes dónde dejé las llaves del coche?"]:
    print(sentence, "->", to_emojis(sentence))

## 9. Run it on the dev set

The model writes an emoji sentence for each of the 300 dev sentences (100 per language). Both runs do all three languages: for control, Spanish shows what the model does in a language it never trained on.
The file goes back to your Mac, where it is scored the same way as the v1 baseline.

In [ ]:
import csv

with open(DEV_FILE, newline="", encoding="utf-8") as f:
    dev = list(csv.DictReader(f))
assert len(dev) == 300, f"Expected 300 dev sentences, found {len(dev)}: is this the step 8 dev.csv?"

for i, row in enumerate(dev, 1):
    row["output"] = to_emojis(row["sentence"])
    if i % 50 == 0:
        print(f"{i}/{len(dev)}")

outputs_file = f"dev_outputs_{RUN_NAME}.csv"
with open(outputs_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "lang", "source", "type", "sentence", "output"], extrasaction="ignore")
    writer.writeheader()
    writer.writerows(dev)
print("Saved", outputs_file)

## 10. Save your results

**Download this zip.** A copy also goes to your Google Drive. It contains the best adapter, the dev outputs, the loss history and the settings.

Put it in `10_v2_fine_tuning/runs/` in your repo and tell me when it's there. Then disconnect the runtime.

In [ ]:
import shutil

run_dir = f"run_{RUN_NAME}"
os.makedirs(run_dir, exist_ok=True)
model.save_pretrained(f"{run_dir}/adapter")
tokenizer.save_pretrained(f"{run_dir}/adapter")
shutil.copy(outputs_file, run_dir)
with open(f"{run_dir}/loss_history.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=1)
with open(f"{run_dir}/settings.json", "w") as f:
    json.dump({**run_config, "best_epoch": best["epoch"], "best_eval_loss": best["eval_loss"],
               "train_examples": len(train_data), "validation_examples": len(validation_data)}, f, indent=1)

archive = shutil.make_archive(run_dir, "zip", run_dir)
shutil.copy(archive, DRIVE_DIR)
print("Downloading", archive)
files.download(archive)